In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from IPython.display import display
import urllib.request

## Khởi tạo Julia và tải dữ liệu benchmark

In [ ]:
!curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz -o julia.tar.gz
!tar -xzf julia.tar.gz -C /usr --strip-components 1

In [ ]:
!git clone https://github.com/cosude/Data-Mining-FIM-Group-7.git
%cd Data-Mining-FIM-Group-7
!julia --project=. -e 'using Pkg; Pkg.instantiate()'
!julia --project=. download_benchmark.jl

## Chương 1: Cài đặt và kiểm thử

In [ ]:
!julia --project=. tests/runtests.jl

## Chương 2: Thực nghiệm và đánh giá

### 2.1. Kết quả đối chiếu với SPMF

In [ ]:
!julia --project=. experiments/exp1_correctness.jl

### 2.2. Mối quan hệ giữa minsupport và thời gian chạy

In [ ]:
!julia --project=. experiments/exp2_runtime_vs_minsup.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

csv_path = "data/results/exp2_runtime_vs_minsup/exp2_runtime.csv"
out_dir = "data/results/exp2_runtime_vs_minsup"
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(csv_path)
datasets = df['Dataset'].unique()

for dataset in datasets:
    data = df[df['Dataset'] == dataset]
    
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(data['Minsup'], data['SPMF_Time_s'], 
             label="SPMF (Java)", marker='o', lw=2, color='blue', linestyle='-')
             
    plt.plot(data['Minsup'], data['Base_Time_s'], 
             label="Basic (Julia)", marker='s', lw=2, color='red', linestyle='--')
             
    plt.plot(data['Minsup'], data['Opt_Time_s'], 
             label="Optimized (Julia)", marker='D', lw=2, color='green', linestyle=':')
    
    plt.gca().invert_xaxis()
    
    plt.title(f"Runtime vs. Minsup ({dataset.capitalize()})")
    plt.xlabel("Minimum Support")
    plt.ylabel("Runtime (Seconds)")
    
    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
    
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    plt.savefig(os.path.join(out_dir, f"plot_runtime_{dataset}.png"), bbox_inches="tight")
    plt.show()

### 2.3. Mối quan hệ giữa số lượng tập phổ biến và minsupport

In [ ]:
!julia --project=. experiments/exp3_itemset_count.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

csv_path = "data/results/exp3_itemset_count/exp3_itemset_counts.csv"
out_dir = "data/results/exp3_itemset_count"
os.makedirs(out_dir, exist_ok=True)

df_counts = pd.read_csv(csv_path)
datasets = df_counts['Dataset'].unique()

for dataset in datasets:
    data = df_counts[df_counts['Dataset'] == dataset]
    ds_type = data['Type'].iloc[0]
    
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(data['Minsup'], data['Itemset_Count'], 
             label="Itemset Count", marker='o', lw=2.5, color='blue', linestyle='-')
    
    plt.gca().invert_xaxis()
    
    plt.yscale('log')
    
    plt.title(f"Output Size vs. Minsup\n({dataset.capitalize()} - {ds_type})")
    plt.xlabel("Minimum Support")
    plt.ylabel("Number of Itemsets (Log Scale)")
    
    plt.grid(True, which='major', alpha=0.5)
    plt.grid(True, which='minor', alpha=0.2, linestyle=':')
    
    plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
    plt.tight_layout()
    
    plt.savefig(os.path.join(out_dir, f"plot_count_{dataset}.png"), bbox_inches="tight")
    plt.show()

### 2.4. So sánh hiệu năng bộ nhớ

In [ ]:
!julia --project=. experiments/exp4_memory.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

dir_path = "data/results/exp4_memory"
csv_path = os.path.join(dir_path, "exp4_memory.csv")
os.makedirs(dir_path, exist_ok=True)

df = pd.read_csv(csv_path)

base = df[df['Impl'] == 'base'].copy().reset_index(drop=True)
opt = df[df['Impl'] == 'opt'].copy().reset_index(drop=True)

names = base['Dataset'].values
x = np.arange(len(names))
width = 0.4

metrics = [
    ('Peak_RSS_MB', 'Peak RAM (whole process)', 'plot_peak_rss.png'),
    ('Delta_RSS_MB', 'Extra peak RAM used by mining (Δ RSS)', 'plot_delta_rss.png'),
    ('Alloc_MB', 'Total allocated memory (GC pressure)', 'plot_alloc.png')
]

for col, ttl, fname in metrics:
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.bar(x - width/2, base[col], width=width, label="Basic implementation", color='#d62728', alpha=0.8)
    plt.bar(x + width/2, opt[col], width=width, label="Optimized implementation", color='#2ca02c', alpha=0.8)
    
    plt.xticks(x, names)
    plt.title(ttl)
    plt.ylabel("MB")
    plt.legend(loc="upper left")
    plt.grid(True, axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(dir_path, fname), bbox_inches="tight")
    plt.show()

base['Tree_Nodes_Safe'] = base['Tree_Nodes'].apply(lambda n: max(n, 1))
bytes_per_node = (base['Tree_MB'] * (1024**2)) / base['Tree_Nodes_Safe']

plt.figure(figsize=(9, 5), dpi=300)
plt.bar(x, bytes_per_node, width=0.6, label="Basic FP-tree", color='#1f77b4', alpha=0.8)

plt.xticks(x, names)
plt.title("Memory per FP-tree node")
plt.ylabel("Bytes / node")
plt.legend(loc="upper right")
plt.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_bytes_per_node.png"), bbox_inches="tight")
plt.show()

print("\n" + "="*80)
print("BẢNG TÓM TẮT THÔNG SỐ (BASE IMPLEMENTATION)")
print("="*80)
base_summary = base[['Dataset', 'Minsup', 'Tree_Nodes', 'Tree_MB', 'DB_MB', 'Peak_RSS_MB', 'Delta_RSS_MB']]
display(base_summary)

print("\n" + "="*80)
print("BẢNG THAM KHẢO SPMF (JVM HEAP)")
print("="*80)
spmf = df[df['Impl'] == 'spmf'].dropna(axis=1, how='all')
display(spmf)

### 2.5. Khả năng mở rộng đối với các mức kích thước cơ sở dữ liệu

In [ ]:
!julia --project=. experiments/exp5_scalability.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

dir_path = "data/results/exp5_scalability"
csv_path = os.path.join(dir_path, "exp5_scalability.csv")
os.makedirs(dir_path, exist_ok=True)

df = pd.read_csv(csv_path)

def linear_fit(x, y):
    x_mean, y_mean = np.mean(x), np.mean(y)
    a = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean)**2)
    b = y_mean - a * x_mean
    
    y_pred = a * x + b
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y_mean)**2)
    r2 = 1 - (ss_res / ss_tot)
    
    return a, b, r2

datasets = df['Dataset'].unique()

for dataset in datasets:
    data = df[df['Dataset'] == dataset].copy()
    
    n = data['Transactions'] / 1000
    minsup_val = data['Minsup'].iloc[0]
    
    a, b, r2 = linear_fit(n, data['Base_Time_s'])
    
    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(n, data['Base_Time_s'], label="Basic implementation", marker='s', lw=2, color='#d62728')
    plt.plot(n, data['Opt_Time_s'], label="Optimized implementation", marker='D', lw=2, color='#2ca02c')
    
    if not data['SPMF_Time_s'].isna().all():
        plt.plot(n, data['SPMF_Time_s'], label="SPMF", marker='o', lw=2, color='#1f77b4')
        
    plt.plot(n, data['Base_Build_s'], label="Basic: FP-tree construction only", marker='^', ls=':', color='purple')
    plt.plot(n, a * n + b, label=f"Linear fit (R² = {r2:.3f})", ls='--', color='gray')
    
    plt.title(f"Scalability ({dataset.capitalize()}, minsup = {minsup_val})")
    plt.xlabel("Number of transactions (K)")
    plt.ylabel("Runtime (Seconds)")
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(dir_path, f"plot_scalability_{dataset}.png"), bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(9, 5), dpi=300)
    
    plt.plot(n, data['Base_Time_s'] / n, label="Basic", marker='s', lw=2, color='#d62728')
    plt.plot(n, data['Opt_Time_s'] / n, label="Optimized", marker='D', lw=2, color='#2ca02c')
    
    plt.title(f"Runtime per 1K transactions ({dataset.capitalize()})")
    plt.xlabel("Number of transactions (K)")
    plt.ylabel("Seconds / 1K transactions")
    
    plt.ylim(bottom=0)
    
    plt.legend(loc="upper left")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(dir_path, f"plot_time_per_trans_{dataset}.png"), bbox_inches="tight")
    plt.show()

print("All Scalability plots (Experiment 5) have been saved successfully!")

### 2.6. Ảnh hưởng của độ dài trung bình của giao dịch

In [ ]:
!julia --project=. experiments/exp6_avg_length.jl

In [ ]:
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

dir_path = "data/results/exp6_avg_length"
csv_path = os.path.join(dir_path, "exp6_avg_length.csv")
os.makedirs(dir_path, exist_ok=True)

df = pd.read_csv(csv_path)

T = df['Actual_Avg_Len']
minsup_val = df['Minsup'].iloc[0]

plt.figure(figsize=(9, 5), dpi=300)

plt.plot(T, df['Base_Time_s'], label="Basic implementation", marker='s', lw=2, color='#d62728', linestyle='--')
plt.plot(T, df['Opt_Time_s'], label="Optimized implementation", marker='D', lw=2, color='#2ca02c', linestyle=':')

if not df['SPMF_Time_s'].isna().all():
    plt.plot(T, df['SPMF_Time_s'], label="SPMF", marker='o', lw=2, color='#1f77b4', linestyle='-')

plt.plot(T, df['Base_Build_s'], label="Basic: FP-tree construction only", marker='^', lw=2, color='purple', linestyle=':')

plt.title(f"Runtime vs. Avg. Length (minsup = {minsup_val})")
plt.xlabel("Average transaction length")
plt.ylabel("Runtime (Seconds)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_runtime_vs_length.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5), dpi=300)

plt.plot(T, df['Itemsets'], label="Frequent itemsets", marker='o', lw=2, color='blue', linestyle='-')
plt.plot(T, df['Tree_Nodes'], label="FP-tree nodes", marker='s', lw=2, color='#ff7f0e', linestyle='--')

plt.title("Itemsets and tree size vs. Avg. Length")
plt.xlabel("Average transaction length")
plt.ylabel("Count (log scale)")
plt.yscale('log')

plt.legend(loc="lower right")
plt.grid(True, which='major', alpha=0.5)
plt.grid(True, which='minor', alpha=0.2, linestyle=':')

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_count_vs_length.png"), bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5), dpi=300)

plt.plot(df['Itemsets'], df['Base_Time_s'], label="Basic", marker='s', lw=2, color='#d62728', linestyle='--')
plt.plot(df['Itemsets'], df['Opt_Time_s'], label="Optimized", marker='D', lw=2, color='#2ca02c', linestyle=':')

plt.title("Runtime vs. Number of Frequent Itemsets")
plt.xlabel("Frequent itemsets")
plt.ylabel("Runtime (Seconds)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(dir_path, "plot_runtime_vs_itemsets.png"), bbox_inches="tight")
plt.show()

print("\n" + "="*80)
print("BẢNG DỮ LIỆU THỰC NGHIỆM 6 (AVG LENGTH)")
print("="*80)
display(df)

## Chương 3: Ứng dụng thực tế

### 3.1. Ứng dụng 1: Phân tích Giỏ hàng

Để thực hiện phân tích giỏ hàng, tập dữ liệu được sử dụng là **Groceries** (dữ liệu bán lẻ thực tế) có định dạng CSV. Mỗi dòng tương ứng với một giao dịch, gồm các item biểu diễn dưới dạng chuỗi (VD: "banana"), ngăn cách nhau bởi dấu phẩy.

Do thuật toán FP-growth của nhóm cài đặt sử dụng cấu trúc dữ liệu có thuộc tính nhãn item được biểu diễn bởi số nguyên, kỹ thuật mã hóa các item từ **Groceries** sau được áp dụng:
1. Duyệt qua từng dòng của tập dữ liệu, lập chỉ mục ánh xạ một item thành một số nguyên duy nhất thông qua một bảng băm.
2. Chuyển đổi từng dòng giao dịch thành mảng số nguyên theo định dạng của SPMF.
3. Kết quả khai thác luật kết hợp sẽ chuyển đổi cách biểu diễn số nguyên về lại chuỗi tên item.

In [ ]:
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv"
urllib.request.urlretrieve(url, "groceries.csv")

item_to_id = {}
current_id = 1

with open("groceries.csv", "r", encoding="utf-8") as f, \
     open("encoded_groceries.txt", "w", encoding="utf-8") as data_out, \
     open("mapping.csv", "w", encoding="utf-8") as map_out:
    
    map_out.write("ID,Item\n")
    for line in f:
        items = [i.strip() for i in line.strip().split(',') if i.strip()]
        if not items: continue
        
        encoded_transaction = []
        for item in items:
            if item not in item_to_id:
                item_to_id[item] = current_id
                map_out.write(f"{current_id},{item}\n")
                current_id += 1
            encoded_transaction.append(str(item_to_id[item]))
            
        data_out.write(" ".join(encoded_transaction) + "\n")
print(f"Encoded sucessfully {len(item_to_id)} items.")

In [ ]:
!julia --project=. applications/app1_market_basket.jl